In [ ]:
# ── CELL 0: Imports + Dataset Setup ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)

# Shared dataset
n = 500
salaries    = np.random.exponential(60000, n)          # right-skewed
scores      = np.random.normal(75, 10, n).clip(40,100) # roughly normal
ages        = np.random.randint(22, 65, n).astype(float)
experience  = np.random.randint(0, 40, n).astype(float)
promoted    = np.random.choice([0, 1], n, p=[0.75, 0.25])

df = pd.DataFrame({
    'salary': salaries, 'score': scores,
    'age': ages, 'experience': experience,
    'promoted': promoted
})
print('Dataset shape:', df.shape)
print(df.describe().round(2))

In [ ]:
# ── CELL 1: STEP 1 — Mean, Median, Mode ───────────────────────────────────────
data = np.array([10, 20, 20, 30, 40, 1000])  # 1000 = outlier

mean   = np.mean(data)
median = np.median(data)
mode   = stats.mode(data, keepdims=True)

print(f'Data:   {data}')
print(f'Mean:   {mean:.1f}  ← pulled toward outlier')
print(f'Median: {median:.1f}  ← robust to outlier')
print(f'Mode:   {mode.mode[0]}')

# Salary column comparison
print('\nSalary column:')
print(f'  Mean:   ${df["salary"].mean():,.0f}')
print(f'  Median: ${df["salary"].median():,.0f}')
print(f'  Skew:   {df["salary"].skew():.2f}  (>1 = use median for central tendency)')

In [ ]:
# ── CELL 2: STEP 2 — Variance and Standard Deviation ──────────────────────────
a = np.array([50, 50, 50, 50, 50])    # zero spread
b = np.array([10, 30, 50, 70, 90])    # wide spread

for name, arr in [('a (no spread)', a), ('b (wide spread)', b)]:
    print(f'{name}: var={np.var(arr):.1f}  std={np.std(arr):.1f}')

# Population vs sample std
sample = np.array([10, 20, 30, 40, 50])
print(f'\nPopulation std (ddof=0): {np.std(sample, ddof=0):.4f}')
print(f'Sample std     (ddof=1): {np.std(sample, ddof=1):.4f}  ← use for samples')
print(f'pandas .std()  (ddof=1): {pd.Series(sample).std():.4f}')

In [ ]:
# ── CELL 3: STEP 3 — Z-Score Normalization (StandardScaler) ───────────────────
from sklearn.preprocessing import StandardScaler

raw = df['score'].values.reshape(-1, 1)

# Manual z-score
z_manual = (df['score'] - df['score'].mean()) / df['score'].std()

# sklearn StandardScaler
scaler = StandardScaler()
z_sklearn = scaler.fit_transform(raw).flatten()

print('Manual z-score stats:')
print(f'  mean={z_manual.mean():.6f}  std={z_manual.std():.4f}')

print('sklearn z-score stats:')
print(f'  mean={z_sklearn.mean():.6f}  std={z_sklearn.std():.4f}')

# Meaning of z-scores
example = (85 - df['score'].mean()) / df['score'].std()
print(f'\nA score of 85 → z = {example:.2f}  ({example:.2f} std above mean)')

In [ ]:
# ── CELL 4: STEP 4 — Percentiles and IQR (RobustScaler) ───────────────────────
from sklearn.preprocessing import RobustScaler

sal = df['salary']

p25 = np.percentile(sal, 25)
p50 = np.percentile(sal, 50)
p75 = np.percentile(sal, 75)
iqr = p75 - p25

print(f'25th percentile (Q1): ${p25:,.0f}')
print(f'50th percentile (Q2): ${p50:,.0f}')
print(f'75th percentile (Q3): ${p75:,.0f}')
print(f'IQR = Q3-Q1:          ${iqr:,.0f}')

lower_fence = p25 - 1.5 * iqr
upper_fence = p75 + 1.5 * iqr
outliers    = sal[(sal < lower_fence) | (sal > upper_fence)]
print(f'Outlier fences: [{lower_fence:,.0f}, {upper_fence:,.0f}]')
print(f'Outlier count: {len(outliers)}')

# RobustScaler (uses median + IQR, not mean + std)
rs = RobustScaler()
sal_robust = rs.fit_transform(sal.values.reshape(-1, 1)).flatten()
print(f'\nRobustScaler output median: {np.median(sal_robust):.4f} (≈0)')

In [ ]:
# ── CELL 5: STEP 5 — Normal Distribution + 68-95-99.7 Rule ────────────────────
mu, sigma = 100, 15   # e.g., IQ scores

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
y = stats.norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, 'k', linewidth=2)

colors   = ['#4CAF50', '#2196F3', '#9C27B0']
sigmas   = [1, 2, 3]
percents = [68, 95, 99.7]

for s, pct, c in zip(sigmas, percents, colors):
    ax.fill_between(x, y,
        where=(x >= mu - s*sigma) & (x <= mu + s*sigma),
        alpha=0.3, color=c, label=f'±{s}σ = {pct}%')

ax.axvline(mu, color='red', linestyle='--', label=f'μ={mu}')
ax.set_title('Normal Distribution — 68-95-99.7 Rule')
ax.set_xlabel('Value')
ax.legend()
plt.tight_layout()
plt.show()

# Stats.norm utilities
print(f'P(x < 115) = {stats.norm.cdf(115, mu, sigma):.4f}')
print(f'P(x > 115) = {stats.norm.sf(115, mu, sigma):.4f}')
print(f'Value at 90th percentile = {stats.norm.ppf(0.90, mu, sigma):.1f}')

In [ ]:
# ── CELL 6: STEP 6 — Other Distributions ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Exponential
axes[0,0].hist(np.random.exponential(60000, 1000), bins=50, color='tomato', edgecolor='white')
axes[0,0].set_title('Exponential — Salary / Wait Time\n(right-skewed → use log transform)')

# Binomial
binom_data = np.random.binomial(n=10, p=0.3, size=1000)
axes[0,1].hist(binom_data, bins=range(12), color='steelblue', edgecolor='white', align='left')
axes[0,1].set_title('Binomial — n=10, p=0.3\n(successes in 10 trials)')

# Poisson
poisson_data = np.random.poisson(lam=3, size=1000)
axes[1,0].hist(poisson_data, bins=range(12), color='seagreen', edgecolor='white', align='left')
axes[1,0].set_title('Poisson — λ=3\n(events per unit time)')

# Uniform
axes[1,1].hist(np.random.uniform(0, 1, 1000), bins=30, color='gold', edgecolor='white')
axes[1,1].set_title('Uniform — [0, 1]\n(no preference, e.g. random weight init)')

plt.suptitle('Common Distributions in ML', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 7: STEP 7 — Covariance and Pearson Correlation ───────────────────────
x_arr = np.array([1, 2, 3, 4, 5], dtype=float)
y_arr = np.array([2, 4, 5, 4, 5], dtype=float)

# Covariance
cov = np.cov(x_arr, y_arr)
print('Covariance matrix:')
print(cov.round(2))
print(f'Cov(x,y) = {cov[0,1]:.2f}  (positive = move together)')

# Pearson correlation
r, p = stats.pearsonr(x_arr, y_arr)
print(f'\nPearson r = {r:.4f}  (scale-free, -1 to +1)')
print(f'p-value   = {p:.4f}')

# Correlation matrix from DataFrame
print('\nDataFrame correlation matrix:')
print(df[['salary','score','age','experience']].corr().round(3))

In [ ]:
# ── CELL 8: STEP 8 — Central Limit Theorem ────────────────────────────────────
population = np.random.exponential(scale=2, size=100_000)  # skewed

sample_sizes  = [5, 30, 100]
n_experiments = 1000

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Population
axes[0].hist(population[:2000], bins=50, color='tomato', edgecolor='white')
axes[0].set_title('Population\n(exponential, skewed)')

# Sample means for each n
for i, n_s in enumerate(sample_sizes):
    means = [np.mean(np.random.choice(population, n_s)) for _ in range(n_experiments)]
    axes[i+1].hist(means, bins=40, color='steelblue', edgecolor='white')
    axes[i+1].set_title(f'Sample Means (n={n_s})\n→ CLT kicks in')

plt.suptitle('Central Limit Theorem — Skewed Population → Normal Sample Means', fontsize=12)
plt.tight_layout()
plt.show()

print('CLT: No matter the population shape, sample means follow a Normal distribution')
print('This is WHY StandardScaler + Normal assumptions work in most ML algorithms.')

In [ ]:
# ── CELL 9: STEP 9 — Hypothesis Testing + p-values ────────────────────────────
# Question: Do promoted employees score significantly higher?
promoted_scores     = df[df['promoted'] == 1]['score']
not_promoted_scores = df[df['promoted'] == 0]['score']

print(f'Promoted     mean score: {promoted_scores.mean():.2f}  (n={len(promoted_scores)})')
print(f'Not promoted mean score: {not_promoted_scores.mean():.2f}  (n={len(not_promoted_scores)})')

# Two-sample t-test
t_stat, p_value = stats.ttest_ind(promoted_scores, not_promoted_scores)
alpha = 0.05

print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.4f}')
print(f'alpha:       {alpha}')

if p_value < alpha:
    print('\nResult: REJECT H₀ — scores differ significantly (p < 0.05)')
else:
    print('\nResult: FAIL TO REJECT H₀ — no significant difference (p ≥ 0.05)')

# Chi-squared test for categorical
contingency = pd.crosstab(df['promoted'],
                           pd.cut(df['age'], bins=[20,35,50,70], labels=['Young','Mid','Senior']))
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'\nChi-squared test (promoted vs age_group):')
print(f'  chi2={chi2:.4f}  p={p_chi:.4f}  dof={dof}')

In [ ]:
# ── CELL 10: STEP 10 — Confidence Intervals ───────────────────────────────────
sample = df['score'].sample(50, random_state=42)

# Parametric CI (t-distribution)
mean   = sample.mean()
se     = stats.sem(sample)           # standard error
ci     = stats.t.interval(0.95, df=len(sample)-1, loc=mean, scale=se)

print(f'Sample mean: {mean:.2f}')
print(f'Standard error: {se:.4f}')
print(f'95% CI (parametric): [{ci[0]:.2f}, {ci[1]:.2f}]')
print(f'Width: {ci[1]-ci[0]:.2f} — if model reports a 95% CI this wide, that is the uncertainty')

# Bootstrap CI (distribution-free)
n_boot    = 5000
boot_means = [np.mean(np.random.choice(sample, len(sample))) for _ in range(n_boot)]
boot_ci    = np.percentile(boot_means, [2.5, 97.5])

print(f'\n95% CI (bootstrap, n={n_boot}): [{boot_ci[0]:.2f}, {boot_ci[1]:.2f}]')
print('Bootstrap CI makes no distribution assumption — preferred for ML metrics')

In [ ]:
# ── CELL 11: STEP 11 — Scaler Selection Decision ──────────────────────────────
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

# Inject outliers into a copy of salary
sal = df['salary'].copy()
sal.iloc[:5] = 1_000_000   # extreme outliers
X  = sal.values.reshape(-1, 1)

scalers = {
    'StandardScaler (z-score)': StandardScaler(),
    'RobustScaler (IQR)':       RobustScaler(),
    'MinMaxScaler [0,1]':       MinMaxScaler(),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, scaler) in zip(axes, scalers.items()):
    scaled = scaler.fit_transform(X).flatten()
    ax.hist(scaled, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'{name}\nmean={scaled.mean():.2f} std={scaled.std():.2f}')

plt.suptitle('Scaler Comparison — Data with Extreme Outliers', fontsize=12)
plt.tight_layout()
plt.show()

print('\nScaler decision guide:')
print('  Normal dist, no outliers  → StandardScaler')
print('  Outliers present          → RobustScaler')
print('  Neural nets, image pixels → MinMaxScaler')
print('  Tree-based models         → no scaling needed')

In [ ]:
# ── CELL 12: Full Statistics Summary Function ──────────────────────────────────
def stats_summary(series, name='column'):
    print('=' * 55)
    print(f'Stats Summary: {name}')
    print('=' * 55)
    print(f'  Mean:     {series.mean():.4f}')
    print(f'  Median:   {series.median():.4f}')
    print(f'  Std:      {series.std():.4f}')
    print(f'  Variance: {series.var():.4f}')
    print(f'  Skew:     {series.skew():.4f}')
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    print(f'  IQR:      {iqr:.4f}')
    n_out = ((series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)).sum()
    print(f'  Outliers: {n_out}')
    if abs(series.skew()) > 1:
        print('  ⚠ Highly skewed → consider log transform + RobustScaler')
    else:
        print('  ✓ Near-normal → StandardScaler is fine')

stats_summary(df['salary'], 'salary')
print()
stats_summary(df['score'], 'score')